# RiskLens agent: ask Nemotron why the allowance moved

The workflow notebook (`RiskLens_CECL_Workflow.ipynb`) books the official CECL runs into `Risk Lens/data/`. This notebook puts an **agent on top of those runs**. Nemotron 3.5 Lightning (30B total / 3B active parameters) plans the investigation and calls RiskLens's deterministic tools. A grounding check then verifies that every number in its answer came from a tool result.

**Pick a backend** (cell 2):

| Backend | What you need | GPU in Colab? |
|---|---|---|
| `nvidia` | Colab secret `NVIDIA_API_KEY` (free key from build.nvidia.com) | No: NVIDIA hosts the model |
| `vllm` | Colab secret `BREV_VLLM_URL`, e.g. `http://<brev-host>:8000/v1`, from your Brev instance running vLLM | No: the model runs on Brev |
| `mock` | nothing | No: deterministic planner, for testing without a model |

Colab secrets also need `GITHUB_TOKEN` (repo read access), the same as the workflow notebook.

## 1 · Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT = Path(os.environ.get("RISKLENS_DRIVE_ROOT", "/content/drive/MyDrive/Risk Lens"))
REPO_DIR = Path(os.environ.get("RISKLENS_REPO", "/content/risklens-cecl"))
REPO_URL = "github.com/Luizkauffmann/risklens-cecl.git"

def sh(*cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr[-2000:])
    return r.stdout

def secret(name, default=None):
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except userdata.SecretNotFoundError:
            return default
        except userdata.NotebookAccessError:
            raise SystemExit(f"Secret {name} exists but this notebook can't read it: open the key icon and switch on 'Notebook access'.")
    return os.environ.get(name, default)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    token = secret("GITHUB_TOKEN")
    if not token:
        raise SystemExit("Add a Colab secret named GITHUB_TOKEN (see the workflow notebook for how to create it), then re-run.")
    url = f"https://{token.strip()}@{REPO_URL}"
    try:
        sh("git", "ls-remote", "-q", url, "HEAD")
    except RuntimeError:
        raise SystemExit("GitHub refused GITHUB_TOKEN: check it hasn't expired and has Contents: Read-only on risklens-cecl.")
    if not (REPO_DIR / ".git").exists():
        sh("git", "clone", "-q", "--depth", "1", url, str(REPO_DIR))
        sh("git", "-C", str(REPO_DIR), "remote", "set-url", "origin", f"https://{REPO_URL}")
    else:
        sh("git", "-C", str(REPO_DIR), "pull", "-q", url, "main")
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt"))

os.environ.update({
    "RISKLENS_CONFIG_DIR": str(DRIVE_ROOT / "config"),
    "RISKLENS_DATA_DIR": str(DRIVE_ROOT / "data"),
    "RISKLENS_OUTPUT_DIR": str(DRIVE_ROOT / "outputs"),
    "RISKLENS_GROUND_TRUTH_DIR": str(DRIVE_ROOT / "evaluation"),
})
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

import shutil
(DRIVE_ROOT / "config").mkdir(parents=True, exist_ok=True)
for f in (REPO_DIR / "config").glob("*.yaml"):          # new config files (e.g. agents.yaml updates) from the repo
    if f.name == "agents.yaml" or not (DRIVE_ROOT / "config" / f.name).exists():
        shutil.copy(f, DRIVE_ROOT / "config" / f.name)

from src.utils import config
config.reload()
from src.data.loader import DataStore
from src.pipeline import cecl_run
store = DataStore()
if not (store.dir / "allowance_summary.csv").exists():   # workflow notebook not run yet: seed + book quickly
    cecl_run.seed_source_data()
    for d in config.settings()["reporting_dates"]:
        cecl_run.run_official(d)
print("data:", store.dir, "| periods:", DataStore().available_dates(), "| code:",
      sh("git", "-C", str(REPO_DIR), "log", "-1", "--format=%h %s").strip())

## 2 · Connect to Nemotron

In [ ]:
BACKEND = os.environ.get("RISKLENS_AGENT_BACKEND", "nvidia")   # "nvidia" | "vllm" | "mock"
ENABLE_THINKING = True                                          # Nemotron reasoning mode (slower, better planning)

overrides = {"provider": BACKEND, "enable_thinking": ENABLE_THINKING}
if BACKEND == "nvidia":
    os.environ["NVIDIA_API_KEY"] = secret("NVIDIA_API_KEY") or ""
elif BACKEND == "vllm":
    overrides["base_url"] = secret("BREV_VLLM_URL")

from src.llm.provider import get_provider
from src.tools import tool_schemas
provider = get_provider(overrides)
print("endpoint:", provider.describe())

# smoke test: one plain turn + one tool call
r = provider.chat([{"role": "user", "content": "Reply with exactly: RiskLens ready"}])
print(f"chat   : {r.content.strip()[:60]!r}  ({r.latency_ms:.0f} ms)")
tools = [s for s in tool_schemas() if s["function"]["name"] == "compare_allowance"]
r = provider.chat([{"role": "user", "content": "How much did the CECL allowance change between 2026-07-31 and 2026-08-31?"}], tools=tools)
print("tools  :", [(t.name, t.arguments) for t in r.tool_calls] or r.content[:200], "| format:", r.raw_tool_format)

## 3 · Ask
`ask()` runs one investigation and shows the answer, a grounding badge, and the trace: every model turn and tool call, with its parameters and `call_id`.

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display
from src.agents.orchestrator import build_orchestrator
from src.tools import ToolCallLog

LOG = ToolCallLog(Path(config.path("outputs")) / "agent_tool_calls.jsonl")
agent = build_orchestrator(provider, log=LOG)
RUNS = []

def ask(question, show_trace=True):
    run = agent.run(question)
    RUNS.append(run)
    g = (run.grounding or {}).get("status", "n/a")
    badge = {"PASS": "✅ every number traced to a tool result", "NO_NUMBERS": "✅ no figures quoted",
             "FAIL": "⚠️ unsupported numbers: " + ", ".join(u["text"] for u in run.grounding["ungrounded"])}.get(g, g)
    display(Markdown(f"**Q: {question}**\n\n{run.answer}\n\n---\n*{run.provider.get('model')} · {run.status} · "
                     f"{run.duration_s}s · {len(run.tool_records)} tool calls · grounding: {badge}*"))
    if show_trace:
        rows = []
        for e in run.events:
            if e["type"] == "llm":
                rows.append({"step": "model", "detail": ", ".join(e["tool_calls"]) or "final answer",
                             "status": e["tool_format"], "call_id": "", "ms": e["latency_ms"]})
            elif e["type"] == "tool":
                rows.append({"step": "tool", "detail": f"{e['name']}({json.dumps(e.get('parameters') or {})})",
                             "status": "duplicate" if e.get("duplicate") else e["status"],
                             "call_id": e.get("call_id"), "ms": e.get("duration_ms")})
                if e.get("error"):
                    rows.append({"step": "  hint", "detail": f"{e['error']['message']} -> {e['error'].get('hint')}",
                                 "status": "", "call_id": "", "ms": None})
            elif e["type"] == "grounding":
                rows.append({"step": "grounding", "detail": ", ".join(e["ungrounded"]) or "all numbers supported",
                             "status": e["status"], "call_id": "", "ms": None})
        with pd.option_context("display.max_colwidth", 140):
            display(pd.DataFrame(rows))
    return run

### The demo questions

In [ ]:
_ = ask("Why did our allowance increase in August?")

In [ ]:
_ = ask("Which macro variable had the largest impact, and in which portfolio?")

In [ ]:
_ = ask("What happened in CRE? Which borrowers should I investigate first?")

In [ ]:
_ = ask("Could this just be because we changed models or scenario weights?")

In [ ]:
_ = ask("Does the attribution reconcile to the reported change?")

### Your own question

In [ ]:
_ = ask("Explain borrower CP_040408: what changed and how much did it add to the allowance?")

## 4 · Tool-calling scorecard
These are the reliability metrics for the **Tool-Calling** track: success rate, arguments repaired, errors recovered, tool calls written as text, and grounding results.

In [ ]:
summary = pd.DataFrame([{"question": r.question[:60], **r.metrics()} for r in RUNS])
display(summary[["question", "tool_calls", "tool_errors", "recovered_after_error", "repaired_arguments",
                 "text_fallback_tool_calls", "revisions", "grounding", "llm_turns", "total_tokens", "duration_s"]])
LOG.stats()